# OCTNet — Multi-Scale Anisotropic CNN from Scratch
**Dataset:** Kermany OCT 2018 — 84K images, 4 classes: CNV / DME / DRUSEN / NORMAL  
**Target:** ≥ 99.5% test accuracy, < 5M parameters, no pretraining  
**Optimized for:** RTX 3060 6GB, Windows/Linux, PyTorch 2.x

## 0. Imports & Device

In [ ]:
import os, random, platform
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# CRITICAL: deterministic=True disables cuDNN autotuner → ~30% slower. Keep False.
# benchmark=True lets cuDNN pick fastest conv algo for your fixed input shape.
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IS_WIN = platform.system() == 'Windows'
print(f'Device: {DEVICE} | Platform: {platform.system()}')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {total_vram:.1f} GB')

## 1. Config

In [ ]:
DATA_ROOT = Path('./OCT2017')   # ← change to your path
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR   = DATA_ROOT / 'val'
TEST_DIR  = DATA_ROOT / 'test'

IMG_SIZE         = 224
BATCH_SIZE       = 128    # 128 @ 224×224 AMP fits in 6GB VRAM; drop to 64 if OOM
NUM_CLASSES      = 4
EPOCHS           = 60
LR               = 3e-4
WEIGHT_DECAY     = 1e-4
LABEL_SMOOTHING  = 0.05
WARMUP_EPOCHS    = 5
GRAD_CLIP        = 1.0
CLASS_NAMES      = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CKPT_PATH        = 'oct_best.pth'

# Windows: num_workers > 0 causes multiprocessing spawn overhead → use 0
# Linux:   4 workers with persistent_workers gives best throughput
NUM_WORKERS = 0 if IS_WIN else 4
PIN_MEMORY  = not IS_WIN and DEVICE.type == 'cuda'  # only useful with async workers
PERSISTENT  = NUM_WORKERS > 0
PREFETCH    = 2 if NUM_WORKERS > 0 else None

print(f'Batch: {BATCH_SIZE} | Workers: {NUM_WORKERS} | pin_memory: {PIN_MEMORY}')

## 2. Augmentation

**OCT-specific choices:**
- ✅ Horizontal flip — retinal layers are horizontal; vertical flip is anatomically invalid
- ✅ Brightness/contrast jitter — scanner gain variation across patients
- ✅ Small affine — eye movement, slight tilt (≤8°, not more)
- ✅ Gaussian noise — OCT speckle artifact
- ❌ No vertical flip, no hue/saturation, no rotation > 10°

In [ ]:
class AddGaussianNoise:
    """Mimics OCT coherence speckle noise."""
    def __init__(self, std=0.02): self.std = std
    def __call__(self, t): return t + torch.randn_like(t) * self.std

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=8, translate=(0.05, 0.03), scale=(0.95, 1.05), shear=3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    AddGaussianNoise(std=0.02),
])

val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

tta_tfs = [
    transforms.Compose([transforms.Grayscale(3), transforms.Resize((IMG_SIZE+16, IMG_SIZE+16)),
                        transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)]),
    transforms.Compose([transforms.Grayscale(3), transforms.Resize((IMG_SIZE+16, IMG_SIZE+16)),
                        transforms.RandomCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)]),
    transforms.Compose([transforms.Grayscale(3), transforms.Resize((IMG_SIZE+16, IMG_SIZE+16)),
                        transforms.RandomCrop(IMG_SIZE), transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)]),
]
print('Transforms defined.')

## 3. DataLoaders

In [ ]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=val_tf)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=val_tf)

# Weighted sampler for class imbalance
cls_counts = Counter(train_ds.targets)
total_n    = sum(cls_counts.values())
sw = torch.tensor([total_n / cls_counts[t] for t in train_ds.targets], dtype=torch.float32)
sampler = WeightedRandomSampler(sw, len(sw), replacement=True)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT,
                 prefetch_factor=PREFETCH)
train_loader = DataLoader(train_ds, sampler=sampler, **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kw)

print(f'Train {len(train_ds):,} | Val {len(val_ds):,} | Test {len(test_ds):,}')
print(f'Class counts: {dict(cls_counts)}')

## 4. Architecture: OCTNet

Three parallel branches per block:
- **3×3 DSConv** — local texture detail
- **5×5 DSConv** — regional context
- **Anisotropic 1×7 + 7×1** — horizontal retinal layer continuity (novel for this dataset)

All branches fused → **CBAM attention** (channel + spatial) → residual add.
Depthwise separable convolutions throughout to stay < 5M params.

In [ ]:
class DSConv(nn.Module):
    """Depthwise separable conv: dw → pw → BN → SiLU."""
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, k, padding=p, groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class AnisoBranch(nn.Module):
    """
    1×7 + 7×1 depthwise separable convolutions.
    Captures horizontal retinal layer continuity (1×7)
    and vertical cross-section disruptions (7×1).
    Outputs summed to keep channel count stable.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.h = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, (1,7), padding=(0,3), groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False))
        self.v = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, (7,1), padding=(3,0), groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False))
        self.bn  = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.h(x) + self.v(x)))


class CBAM(nn.Module):
    """Channel + Spatial attention (Woo et al. 2018)."""
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.ca_mlp = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(inplace=True), nn.Linear(mid, ch, bias=False))
        self.sa_conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)

    def forward(self, x):
        # Channel attention
        avg = x.mean([2,3]); mx = x.amax([2,3])
        ca  = torch.sigmoid(self.ca_mlp(avg) + self.ca_mlp(mx))
        x   = x * ca.unsqueeze(-1).unsqueeze(-1)
        # Spatial attention
        sa  = torch.sigmoid(self.sa_conv(torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], dim=1)))
        return x * sa


class MSBlock(nn.Module):
    """Multi-scale block: 3 branches → concat → CBAM → residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        bc = out_ch // 3
        ex = out_ch - 3 * bc
        self.b3 = DSConv(in_ch, bc + ex, k=3, p=1)
        self.b5 = DSConv(in_ch, bc,      k=5, p=2)
        self.ba = AnisoBranch(in_ch, bc)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.cbam = CBAM(out_ch)
        self.res  = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)) \
                    if in_ch != out_ch else nn.Identity()
        self.act  = nn.SiLU(inplace=True)

    def forward(self, x):
        out = self.cbam(self.bn(torch.cat([self.b3(x), self.b5(x), self.ba(x)], dim=1)))
        return self.act(out + self.res(x))


class OCTNet(nn.Module):
    def __init__(self, num_classes=4, drop=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),  # 112
            nn.Conv2d(32, 64, 3, padding=1, bias=False),           nn.BatchNorm2d(64), nn.SiLU(inplace=True),
        )
        self.s1 = nn.Sequential(MSBlock(64,  96),  nn.MaxPool2d(2), nn.Dropout2d(0.10))  # 56
        self.s2 = nn.Sequential(MSBlock(96,  192), MSBlock(192,192), nn.MaxPool2d(2), nn.Dropout2d(0.15))  # 28
        self.s3 = nn.Sequential(MSBlock(192, 384), MSBlock(384,384), nn.MaxPool2d(2), nn.Dropout2d(0.20))  # 14
        self.s4 = nn.Sequential(MSBlock(384, 512), nn.MaxPool2d(2))  # 7
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.SiLU(inplace=True),
            nn.Dropout(drop), nn.Linear(256, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear) and m.weight is not None:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.s4(self.s3(self.s2(self.s1(self.stem(x))))))

    @property
    def cam_layer(self):
        """Last depthwise conv — target for GradCAM++."""
        return self.s4[0].b3.net[0]


model = OCTNet(NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} ({n_params/1e6:.2f}M)')
assert n_params < 5_000_000, f'Too many params: {n_params:,}'

# torch.compile: free ~15-25% speedup on PyTorch ≥ 2.0, Linux only (unstable on Windows)
if int(torch.__version__.split('.')[0]) >= 2 and not IS_WIN:
    model = torch.compile(model, mode='reduce-overhead')
    print('torch.compile: enabled')
else:
    print('torch.compile: skipped (Windows or PyTorch < 2.0)')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    print(f'VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 5. Loss / Optimizer / Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(ep):
    if ep < WARMUP_EPOCHS: return (ep + 1) / WARMUP_EPOCHS
    p = (ep - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + np.cos(np.pi * p))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Use new torch.amp API (PyTorch ≥ 2.0); fall back to cuda.amp for older versions
_amp_dev = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
try:
    scaler = torch.amp.GradScaler(_amp_dev, enabled=(DEVICE.type == 'cuda'))
    def autocast(): return torch.amp.autocast(_amp_dev, enabled=(DEVICE.type == 'cuda'))
    print('Using torch.amp (new API)')
except AttributeError:
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    def autocast(): return torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda'))
    print('Using torch.cuda.amp (legacy API)')

## 6. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  train', leave=False):
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)   # set_to_none=True: faster than zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  eval ', leave=False):
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


hist = {'tr_loss':[], 'va_loss':[], 'tr_acc':[], 'va_acc':[], 'lr':[]}
best_va = 0.0

print(f'Training {EPOCHS} epochs | batch {BATCH_SIZE} | {DEVICE}')
for ep in range(EPOCHS):
    tr_l, tr_a = train_epoch(model, train_loader, optimizer, criterion, scaler)
    va_l, va_a = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    # Flush VRAM cache every 5 epochs to prevent fragmentation
    if DEVICE.type == 'cuda' and (ep + 1) % 5 == 0:
        torch.cuda.empty_cache()

    for k, v in zip(['tr_loss','va_loss','tr_acc','va_acc','lr'],
                    [tr_l, va_l, tr_a, va_a, optimizer.param_groups[0]['lr']]):
        hist[k].append(v)

    star = ''
    if va_a > best_va:
        best_va = va_a
        # Save raw state_dict even if model is compiled
        raw = model._orig_mod if hasattr(model, '_orig_mod') else model
        torch.save(raw.state_dict(), CKPT_PATH)
        star = ' ★'

    print(f'Ep {ep+1:03d}/{EPOCHS}{star} | '
          f'tr {tr_l:.4f}/{tr_a*100:.2f}% | '
          f'va {va_l:.4f}/{va_a*100:.2f}% | '
          f'lr {optimizer.param_groups[0]["lr"]:.2e}'
          + (f' | VRAM {torch.cuda.memory_allocated()/1e9:.2f}GB' if DEVICE.type == 'cuda' else ''))

print(f'\nBest val acc: {best_va*100:.2f}%')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep_r = range(1, EPOCHS + 1)

axes[0].plot(ep_r, hist['tr_loss'], label='Train'); axes[0].plot(ep_r, hist['va_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(ep_r, [a*100 for a in hist['tr_acc']], label='Train')
axes[1].plot(ep_r, [a*100 for a in hist['va_acc']], label='Val')
axes[1].axhline(99.5, color='red', ls='--', alpha=0.5, label='Target 99.5%')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(ep_r, hist['lr']); axes[2].set_yscale('log')
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Test Evaluation (Standard + TTA)

In [ ]:
# Load best checkpoint into a clean (uncompiled) model
eval_model = OCTNet(NUM_CLASSES).to(DEVICE)
eval_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
eval_model.eval()


@torch.no_grad()
def predict(model, loader):
    preds, probs, labels = [], [], []
    for imgs, lbl in tqdm(loader, desc='  infer', leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(imgs)
        p = torch.softmax(logits, 1).cpu().numpy()
        probs.append(p)
        preds.extend(p.argmax(1))
        labels.extend(lbl.numpy())
    return np.array(preds), np.concatenate(probs), np.array(labels)


@torch.no_grad()
def predict_tta(model, dataset, tfs):
    all_probs = []
    labels_all = [lbl for _, lbl in dataset.samples]
    for tf in tfs:
        dataset.transform = tf
        ld = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        batch_p = []
        for imgs, _ in tqdm(ld, desc='  TTA', leave=False):
            with autocast(): logits = model(imgs.to(DEVICE, non_blocking=True))
            batch_p.append(torch.softmax(logits, 1).cpu().numpy())
        all_probs.append(np.concatenate(batch_p))
    avg = np.mean(all_probs, 0)
    return avg.argmax(1), avg, np.array(labels_all)


if DEVICE.type == 'cuda': torch.cuda.empty_cache()

preds_std, probs_std, labels_te = predict(eval_model, test_loader)
acc_std = accuracy_score(labels_te, preds_std)
print(f'Standard test acc: {acc_std*100:.2f}%')

preds_tta, probs_tta, _ = predict_tta(eval_model, test_ds, tta_tfs)
acc_tta = accuracy_score(labels_te, preds_tta)
print(f'TTA test acc:      {acc_tta*100:.2f}%')

if DEVICE.type == 'cuda': torch.cuda.empty_cache()

In [ ]:
print(classification_report(labels_te, preds_tta, target_names=CLASS_NAMES, digits=4))

lb = label_binarize(labels_te, classes=list(range(NUM_CLASSES)))
aucs = [roc_auc_score(lb[:,i], probs_tta[:,i]) for i in range(NUM_CLASSES)]
for n, a in zip(CLASS_NAMES, aucs): print(f'  AUC {n}: {a:.4f}')
print(f'  Mean AUC: {np.mean(aucs):.4f}')

In [ ]:
cm = confusion_matrix(labels_te, preds_tta)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(axes,
    [cm, cm.astype(float)/cm.sum(1, keepdims=True)],
    ['d', '.3f'], ['counts', 'normalized']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'Confusion Matrix ({title})')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.suptitle(f'OCTNet — TTA acc: {acc_tta*100:.2f}%', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. GradCAM++ Visualization

In [ ]:
class GradCAMpp:
    """GradCAM++ — no external library required."""
    def __init__(self, model, layer):
        self.model = model; self.grads = None; self.acts = None
        layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))

    def __call__(self, img_t, cls=None):
        self.model.eval()
        x = img_t.unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = self.model(x)
        cls = cls if cls is not None else logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, cls].backward()
        g, a = self.grads[0], self.acts[0]            # (C,H,W)
        g2, g3 = g**2, g**3
        alpha = g2 / (2*g2 + (a*g3).sum([1,2], keepdim=True) + 1e-7)
        w = (alpha * F.relu(g)).sum([1,2])             # (C,)
        cam = F.relu((w[:,None,None] * a).sum(0))
        cam = (cam - cam.min()) / (cam.max() + 1e-7)
        conf = torch.softmax(logits, 1)[0].detach().cpu().numpy()
        return cam.cpu().numpy(), cls, conf


cam_fn = GradCAMpp(eval_model, eval_model.cam_layer)
test_ds.transform = val_tf   # restore

fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(12, NUM_CLASSES*3.5))
fig.suptitle('GradCAM++ — OCTNet', fontsize=13, y=1.01)

for ci, cname in enumerate(CLASS_NAMES):
    idxs = [i for i,(_, l) in enumerate(test_ds.samples) if l == ci]
    img_t, true_l = test_ds[idxs[0]]
    for idx in idxs:
        it, tl = test_ds[idx]
        _, pred, _ = cam_fn(it)
        if pred == tl: img_t = it; break

    cam, pred_i, conf = cam_fn(img_t)
    img_np = (img_t.numpy().transpose(1,2,0) * 0.5 + 0.5).clip(0,1).mean(2)
    cam_up = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR))/255.
    overlay = (0.55*np.stack([img_np]*3,2) + 0.45*plt.cm.jet(cam_up)[:,:,:3]).clip(0,1)

    axes[ci,0].imshow(img_np, cmap='gray');    axes[ci,0].set_title(f'Original [{cname}]'); axes[ci,0].axis('off')
    axes[ci,1].imshow(cam_up, cmap='jet');     axes[ci,1].set_title('GradCAM++ heatmap');  axes[ci,1].axis('off')
    axes[ci,2].imshow(overlay)
    axes[ci,2].set_title(f'Pred: {CLASS_NAMES[pred_i]} {"✓" if pred_i==ci else "✗"} ({conf[pred_i]*100:.1f}%)')
    axes[ci,2].axis('off')

plt.tight_layout()
plt.savefig('gradcam.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. SOTA Comparison Table

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'Method':         ['Kermany 2018 (InceptionV3, TL)', 'Lee et al. (from scratch)', 'Sunija et al. (from scratch)', 'MIDNet Mohan et al. (from scratch)', 'OCTNet Ours (from scratch)'],
    'Params':         ['~5M+', 'N/A', 'N/A', 'N/A', f'{n_params/1e6:.2f}M'],
    'Pretrained':     ['✅ ImageNet', '❌', '❌', '❌', '❌'],
    'Test Accuracy':  ['96.60%', '87.63%', '99.69%', '98.86%', f'{acc_tta*100:.2f}% (TTA)'],
})
print(df.to_string(index=False))

## 11. Error Analysis

In [ ]:
test_ds.transform = val_tf
errors = [(i, t, p) for i,(t,p) in enumerate(zip(labels_te, preds_tta)) if t != p]
print(f'Misclassified: {len(errors)}/{len(labels_te)} ({len(errors)/len(labels_te)*100:.2f}%)')
err_pairs = Counter([(CLASS_NAMES[t], CLASS_NAMES[p]) for _,t,p in errors])
print('Error patterns (true → pred):')
for (tc, pc), cnt in err_pairs.most_common(): print(f'  {tc:8s} → {pc:8s}: {cnt}')

In [ ]:
n_show = min(8, len(errors))
if n_show > 0:
    top_errs = sorted(errors, key=lambda x: probs_tta[x[0], x[2]], reverse=True)[:n_show]
    fig, axes = plt.subplots(2, n_show//2, figsize=(16, 6))
    for ax, (idx, tl, pl) in zip(axes.flatten(), top_errs):
        img_np = (test_ds[idx][0].numpy().transpose(1,2,0)*0.5+0.5).clip(0,1).mean(2)
        ax.imshow(img_np, cmap='gray')
        ax.set_title(f'True: {CLASS_NAMES[tl]}\nPred: {CLASS_NAMES[pl]} ({probs_tta[idx,pl]*100:.1f}%)', fontsize=9, color='red')
        ax.axis('off')
    plt.suptitle('Most Confident Errors')
    plt.tight_layout()
    plt.savefig('errors.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Final Summary

In [ ]:
print('='*55)
print('  OCTNet — Final Results')
print('='*55)
print(f'  Params:        {n_params:,} ({n_params/1e6:.2f}M)')
print(f'  From scratch:  YES')
print(f'  Acc (std):     {acc_std*100:.2f}%')
print(f'  Acc (TTA×3):   {acc_tta*100:.2f}%')
print(f'  Macro F1:      {f1_score(labels_te, preds_tta, average="macro")*100:.2f}%')
print(f'  Mean AUC:      {np.mean(aucs):.4f}')
print(f'  Best val acc:  {best_va*100:.2f}%')
print('='*55)